# BÀI THỰC HÀNH SỐ 1 - KHAI PHÁ DỮ LIỆU
## PHẦN 1 - BÀI 2 (TRANG 15-17): TITANIC DATASET
- **Họ và tên:** Bùi Minh Trọng
- **MSSV:** 6451071081
- **Lớp:** CQ.64.CNTT
### Câu 6: Tiền xử lý các thuộc tính thiếu dữ liệu và xóa bỏ các thuộc tính không cần thiết


In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import os

data_candidates = ['titanic.csv', '../titanic.csv', '../../TH1/titanic.csv', '../TH1/titanic.csv']
data_path = next((p for p in data_candidates if os.path.exists(p)), 'titanic.csv')
df = pd.read_csv(data_path)

print("Họ và tên: Bùi Minh Trọng - MSSV: 6451071081")

# 1. Xóa các thuộc tính không cần thiết, rò rỉ dữ liệu hoặc thiếu quá nhiều (Yêu cầu 6)
drop_cols = ['ticket', 'cabin', 'boat', 'body', 'home.dest']
df_clean = df.drop(columns=[col for col in drop_cols if col in df.columns]).copy()

# 2. Điền giá trị thiếu cho thuộc tính tuổi (age) bằng giá trị ngẫu nhiên trong khoảng [mean - std, mean + std]
np.random.seed(42)
mean_age = df_clean['age'].mean()
std_age = df_clean['age'].std()
is_null_age = df_clean['age'].isnull().sum()
rand_age = np.random.randint(int(mean_age - std_age), int(mean_age + std_age), size=is_null_age)
age_slice = df_clean['age'].copy()
age_slice[np.isnan(age_slice)] = rand_age
df_clean['age'] = age_slice.astype(int)

# 3. Điền giá trị thiếu cho cổng lên tàu (embarked) bằng giá trị mode
common_embarked = df_clean['embarked'].mode()[0]
df_clean['embarked'] = df_clean['embarked'].fillna(common_embarked)

# 4. Điền giá trị thiếu cho giá vé (fare) bằng median và chuyển về kiểu nguyên
df_clean['fare'] = df_clean['fare'].fillna(df_clean['fare'].median()).astype(int)

# 5. Biến đổi thuộc tính (Yêu cầu 7):
# 5.1 Trích xuất danh hiệu (title) từ cột name
df_clean['title'] = df_clean['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df_clean['title'] = df_clean['title'].replace(rare_titles, 'Rare')
df_clean['title'] = df_clean['title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
title_mapping = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4, 'Rare': 5}
df_clean['title'] = df_clean['title'].map(title_mapping).fillna(0).astype(int)

# 5.2 Chuyển đổi giới tính (sex) và cổng lên tàu (embarked) thành số
df_clean['sex'] = df_clean['sex'].map({'female': 1, 'male': 0}).astype(int)
df_clean['embarked'] = df_clean['embarked'].map({'S': 0, 'C': 1, 'Q': 2}).astype(int)

# 5.3 Chia độ tuổi thành 8 nhóm cân bằng
age_bins = [-1, 11, 18, 22, 27, 33, 40, 66, 120]
age_labels = [0, 1, 2, 3, 4, 5, 6, 7]
df_clean['age_group'] = pd.cut(df_clean['age'], bins=age_bins, labels=age_labels).astype(int)

# 5.4 Chia giá vé thành 6 nhóm
fare_bins = [-1, 7.91, 14.454, 31, 99, 250, 10000]
fare_labels = [0, 1, 2, 3, 4, 5]
df_clean['fare_group'] = pd.cut(df_clean['fare'], bins=fare_bins, labels=fare_labels).astype(int)

# 6. Tạo các thuộc tính mới (Yêu cầu 8):
# 6.1 Tuổi * Loại vé (age_class)
df_clean['age_class'] = df_clean['age_group'] * df_clean['pclass']
# 6.2 Giá vé trên mỗi người (fare_per_person)
relatives = df_clean['sibsp'] + df_clean['parch']
df_clean['fare_per_person'] = (df_clean['fare'] / (relatives + 1)).astype(int)

print("Kiểm tra lại số lượng giá trị thiếu sau khi tiền xử lý:")
print(df_clean.isnull().sum())
print("\n5 dòng dữ liệu sau tiền xử lý và biến đổi đặc trưng:")
display(df_clean[['survived', 'pclass', 'sex', 'age_group', 'title', 'embarked', 'fare_group', 'age_class', 'fare_per_person']].head())


Họ và tên: Bùi Minh Trọng - MSSV: 6451071081
Kiểm tra lại số lượng giá trị thiếu sau khi tiền xử lý:
pclass             0
survived           0
name               0
sex                0
age                0
sibsp              0
parch              0
fare               0
embarked           0
title              0
age_group          0
fare_group         0
age_class          0
fare_per_person    0
dtype: int64

5 dòng dữ liệu sau tiền xử lý và biến đổi đặc trưng:


,survived,pclass,sex,age_group,title,embarked,fare_group,age_class,fare_per_person
0,1,1,1,4,2,0,4,4,211
1,1,1,0,0,4,0,4,0,37
2,0,1,1,0,2,0,4,0,37
3,0,1,0,4,1,0,4,4,37
4,0,1,1,3,3,0,4,3,37


**Nhận xét Câu 6 (Bao gồm Yêu cầu 6, 7, 8 theo tài liệu PDF):**
- **Xử lý thiếu dữ liệu & lọc cột (Yêu cầu 6):** Đã loại bỏ hoàn toàn các cột không cần thiết hoặc thiếu nhiều (`ticket`, `cabin`, `boat`, `body`, `home.dest`). Đã điền khuyết hợp lý cho `age` (ngẫu nhiên theo chuẩn $\mu \pm \sigma$), `embarked` (mode = 'S'), và `fare` (median).
- **Biến đổi thuộc tính (Yêu cầu 7):** Chuyển đổi `fare` sang số nguyên; trích xuất danh hiệu `title` (Mr: 1, Miss: 2, Mrs: 3, Master: 4, Rare: 5); mã hóa nhị phân `sex` (female: 1, male: 0); số hóa `embarked` (S: 0, C: 1, Q: 2); chia 8 nhóm tuổi và 6 nhóm giá vé theo ngưỡng quy định.
- **Tạo thuộc tính mới (Yêu cầu 8):** Đã tạo thành công `age_class` (tương tác giữa nhóm tuổi và hạng vé) và `fare_per_person` (giá vé thực tế cho từng cá nhân). Bộ dữ liệu hiện tại sẵn sàng tối ưu cho các mô hình học máy.